In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 38
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## RTI Rating Pipeline

**Source:** Global Right to Information (RTI) Rating — Centre for Law and Democracy / Access Info Europe
**Access:** Automated — full scores table parsed directly from the country-data page HTML
**Concept role:** PRIMARY tier 1 — Government transparency (C25, FOI/RTI leg) and Media Freedom (C23)

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| RTI Rating total (0-150) | Govt transparency / Media freedom | Primary tier 1 |
| 7 category sub-scores | (same) | Supporting detail |

### Scope (honest)
- **De jure only** — strength of the legal framework for access to information; does NOT
  measure implementation/practice (CLD's parallel implementation tool is rti-evaluation.org).
- **Cross-sectional snapshot** (current ratings); historical time series deferred.
- ~142 countries WITH an RTI law. Countries with NO RTI law are a separate list (handled below).

In [2]:
# Fetch the RTI Rating country-data page and parse the scores table directly from its HTML.
# Fully automated — the complete table (142 countries x 7 categories + total + law year) is in the page.
RTI_URL = "https://www.rti-rating.org/country-data/"

resp = requests.get(RTI_URL, headers=BROWSER_HEADERS, timeout=60)
resp.raise_for_status()
print(f"Fetched {len(resp.content)/1024:.1f}KB, status {resp.status_code}")

# pandas.read_html extracts all HTML tables; the scores table is the large one.
tables = pd.read_html(io.StringIO(resp.text))
print(f"Tables found: {len(tables)}")
for i, t in enumerate(tables):
    print(f"  Table {i}: shape {t.shape}, columns: {list(t.columns)[:4]}...")

Fetched 277.0KB, status 200
Tables found: 1
  Table 0: shape (142, 13), columns: ['Ranking', 'Country', 'Date', 'Right of Access']...


In [9]:
# Clean the parsed table: keep scores, map country names to ISO3, drop link columns.
rti = tables[0].copy()

# The 7 category columns + Total are the substantive scores
CATEGORY_COLS = ['Right of Access', 'Scope', 'Requesting Procedure',
                 'Exceptions & Refusals', 'Appeals', 'Sanctions & Protections',
                 'Promotional Measures']
# Confirm expected columns exist (raise loudly if the page structure changed)
missing = [c for c in CATEGORY_COLS + ['Total', 'Country', 'Date'] if c not in rti.columns]
if missing:
    raise ValueError(f"Expected columns missing from RTI table: {missing}")

# Keep only the substantive columns (drop Ranking, Law/Report link columns)
rti = rti[['Country', 'Date'] + CATEGORY_COLS + ['Total']].copy()

# Coerce scores to numeric
for c in CATEGORY_COLS + ['Total']:
    rti[c] = pd.to_numeric(rti[c], errors='coerce')

# Map country names to ISO3 via pycountry (with manual fixes for known mismatches)
import pycountry

MANUAL_ISO3 = {
    'South Korea': 'KOR', 'Russia': 'RUS', 'Ivory Coast': 'CIV', 'Bolivia': 'BOL',
    'Venezuela': 'VEN', 'Vietnam': 'VNM', 'Iran': 'IRN', 'Tanzania': 'TZA',
    'Moldova': 'MDA', 'North Macedonia': 'MKD', 'Czech Republic': 'CZE',
    'East Timor': 'TLS', 'Taiwan': 'TWN', 'Kosovo': 'XKX', 'Republic of Belarus': 'BLR',
    'Cape Verde': 'CPV', 'Cook Islands': 'COK', 'Turkey': 'TUR',
    'Democratic Republic of Congo': 'COD', 'Guinea Bissau': 'GNB',
    'Micronesia (Federated States of)': 'FSM',
}

def to_iso3(name):
    n = name.strip()
    if n in MANUAL_ISO3:
        return MANUAL_ISO3[n]
    try:
        return pycountry.countries.lookup(n).alpha_3
    except LookupError:
        return None

rti['country_code'] = rti['Country'].map(to_iso3)

# Report any unmapped countries (so we can add them to MANUAL_ISO3 — a flagged manual step)
unmapped = rti[rti['country_code'].isna()]['Country'].tolist()
print(f"Unmapped countries (need manual ISO3): {unmapped}")
print(f"Mapped: {rti['country_code'].notna().sum()} of {len(rti)}")

Unmapped countries (need manual ISO3): []
Mapped: 142 of 142


In [10]:
import re

# Extract the deficit-file URL dynamically from the page HTML (its filename embeds a date that
# changes on update, so we never hardcode it — we read the current link from the page).
m = re.search(r'href="(https://www\.rti-rating\.org/wp-content/uploads/[^"]*?Countries\.Deficit[^"]*?\.xlsx)"',
              resp.text)
if not m:
    raise RuntimeError("Could not find the deficit-list (no-RTI-law countries) URL on the page")
deficit_url = m.group(1)
print(f"Deficit file URL: {deficit_url}")

# Fetch and load the deficit list (countries with NO RTI law)
dresp = requests.get(deficit_url, headers=BROWSER_HEADERS, timeout=60)
dresp.raise_for_status()
deficit_raw = pd.read_excel(io.BytesIO(dresp.content), engine='openpyxl')
print(f"Deficit file shape: {deficit_raw.shape}, columns: {list(deficit_raw.columns)}")
print(deficit_raw.head(8).to_string())

Deficit file URL: https://www.rti-rating.org/wp-content/uploads/2026/03/Countries.Deficit.26-02.CLD_.xlsx
Deficit file shape: (199, 9), columns: ['Unnamed: 0', 'UN Countries', 'RTI law', 'No RTI Law', 'Non-democractic w/o RTI law*', 'Small w/o RTI law**', 'Other w/o RTI law', 'Non-democratic w/ RTI law*', 'V-Dem Score']
  Unnamed: 0          UN Countries  RTI law  No RTI Law Non-democractic w/o RTI law*  Small w/o RTI law**  Other w/o RTI law  Non-democratic w/ RTI law*  V-Dem Score
0          1           Afganistan       1.0         NaN                          NaN                  NaN                NaN                         1.0         0.02
1          2               Albania      1.0         NaN                          NaN                  NaN                NaN                         NaN         0.38
2          3               Algeria      NaN         1.0                            1                  NaN                NaN                         NaN         0.13
3          4  

In [11]:
# Assemble final RTI panel: rated countries (real scores) + no-law countries (floored total).
# No-law set comes directly from the deficit file's 'No RTI Law' flag column.

# --- Floor value for no-law countries: minimum observed total MINUS one standard deviation ---
# Computed dynamically from the rated data (no hardcoding). This is a documented SCORING CHOICE:
# "no RTI law" is placed below the weakest actual law by one SD of the score distribution. It is an
# assigned floor, NOT a measured value; has_rti_law=0 keeps it identifiable/revisable at metric pass.
min_total = rti['Total'].min()
sd_total = rti['Total'].std()  # sample SD (ddof=1)
no_law_total = min_total - sd_total
print(f"Rated total: min={min_total:.1f}, max={rti['Total'].max():.1f}, SD={sd_total:.1f}")
print(f"No-law floor value (min - 1 SD): {no_law_total:.1f}")

# --- Rated countries frame (has_rti_law = 1, real scores) ---
rated = rti[['country_code', 'Total'] + CATEGORY_COLS + ['Date']].copy()
rated = rated.rename(columns={'Total': 'rti_total', 'Date': 'rti_law_year'})
rated['has_rti_law'] = 1

# --- No-law countries from the deficit file's flag column ---
no_law = deficit_raw[deficit_raw['No RTI Law'] == 1].copy()
no_law['country_code'] = no_law['UN Countries'].map(to_iso3)
unmapped_nl = no_law[no_law['country_code'].isna()]['UN Countries'].tolist()
print(f"\nNo-law countries: {len(no_law)}; unmapped (need manual ISO3): {unmapped_nl}")

Rated total: min=33.0, max=139.0, SD=23.7
No-law floor value (min - 1 SD): 9.3

No-law countries: 54; unmapped (need manual ISO3): []


In [12]:
# Clamp the floor at 0 (a score can't be negative on the 0-150 scale) — robust if a future
# weakest law ever scores below one SD. Documented scoring choice.
no_law_total = max(0.0, min_total - sd_total)
print(f"No-law floor value (clamped >=0): {no_law_total:.1f}")

# Build no-law frame: floored TOTAL, NaN category subscores (no basis to distribute), has_rti_law=0
no_law_clean = pd.DataFrame({'country_code': no_law['country_code'].dropna().unique()})
no_law_clean['rti_total'] = no_law_total
for c in CATEGORY_COLS:
    no_law_clean[c] = pd.NA          # no category-level features for a country with no law
no_law_clean['rti_law_year'] = pd.NA
no_law_clean['has_rti_law'] = 0

# Guard: no country should appear in both rated and no-law sets
overlap = set(rated['country_code']) & set(no_law_clean['country_code'])
if overlap:
    raise ValueError(f"Country in BOTH rated and no-law sets: {overlap}")

# Combine, order columns, sort
panel = pd.concat([rated, no_law_clean], ignore_index=True)
panel = panel[['country_code', 'rti_total'] + CATEGORY_COLS + ['rti_law_year', 'has_rti_law']]
panel = panel.sort_values('country_code').reset_index(drop=True)

print(f"\nFinal panel: {panel.shape}")
print(f"Total countries: {panel['country_code'].nunique()}  "
      f"(rated: {(panel['has_rti_law']==1).sum()}, no-law: {(panel['has_rti_law']==0).sum()})")
print(f"\nScore distribution (rti_total):")
print(panel['rti_total'].describe().round(1))

No-law floor value (clamped >=0): 9.3

Final panel: (196, 11)
Total countries: 196  (rated: 142, no-law: 54)

Score distribution (rti_total):
count    196.0
mean      64.7
std       39.7
min        9.3
25%        9.3
50%       72.0
75%       94.2
max      139.0
Name: rti_total, dtype: float64


In [14]:
# Data currency: derived from the rating year embedded in the data where available.
# The RTI Rating has no single "vintage" field; it's a continuously-updated current snapshot.
# Stamp the retrieval date and note it's a live cross-section.
retrieval_date = datetime.today().strftime("%Y-%m-%d")

output_path = os.path.join(PROCESSED_DIR, "rti_rating_clean.csv")
panel.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {panel.shape}")

n_countries = panel['country_code'].nunique()

update_entry(
    "RTI_RATING",
    last_successful_download_date=retrieval_date,
    data_as_of_date=retrieval_date,   # live cross-section; no vintage field in source
    local_filename="rti_rating_clean.csv",
    latest_available_version="live cross-section",
    notes=("Global RTI Rating (Centre for Law and Democracy / Access Info Europe). PRIMARY tier-1 for "
           "Government transparency (C25 FOI/RTI leg) and Media Freedom (C23). Automated: full scores "
           "table parsed from the country-data page HTML (rti-rating.org/country-data). rti_total (0-150) "
           "+ 7 category sub-scores + rti_law_year. DE JURE only (legal-framework strength, NOT "
           "implementation — CLD's parallel implementation tool is rti-evaluation.org). "
           "142 countries WITH an RTI law (real scores, has_rti_law=1); 54 countries with NO RTI law "
           "(from the deficit-list flag) assigned a floored rti_total = min(observed) - 1 SD, clamped >=0 "
           "(=9.3), with NaN sub-scores and has_rti_law=0. The floor is a documented SCORING CHOICE "
           "(no-law placed below weakest law by one SD), revisable at metric pass via the has_rti_law flag. "
           f"Coverage: {n_countries} countries — strongest in the framework. Cross-section; historical "
           "time series deferred.")
)
print_entry("RTI_RATING")

Written: C:\Users\mjbou\governance-framework\data\processed\rti_rating_clean.csv
Shape: (196, 11)
[download_log] Updated entry for RTI_RATING
  source_id: RTI_RATING
  last_attempted_date: 2026-06-18
  last_successful_download_date: 2026-06-18
  data_as_of_date: 2026-06-18
  local_filename: rti_rating_clean.csv
  latest_available_version: live cross-section
  no_update_reason: nan
  notes: Global RTI Rating (Centre for Law and Democracy / Access Info Europe). PRIMARY tier-1 for Government transparency (C25 FOI/RTI leg) and Media Freedom (C23). Automated: full scores table parsed from the country-data page HTML (rti-rating.org/country-data). rti_total (0-150) + 7 category sub-scores + rti_law_year. DE JURE only (legal-framework strength, NOT implementation — CLD's parallel implementation tool is rti-evaluation.org). 142 countries WITH an RTI law (real scores, has_rti_law=1); 54 countries with NO RTI law (from the deficit-list flag) assigned a floored rti_total = min(observed) - 1 SD, cl